In [2]:
import pandas as pd
import numpy as np

clients = pd.read_csv("data/raw/clients.csv", parse_dates=["join_date"])
trades = pd.read_csv("data/raw/trades.csv", parse_dates=["date"])
fx_daily = pd.read_csv("data/raw/fx_daily.csv", parse_dates=["date"])

print(f"\nclients df = {clients.shape}")
clients.info()
print(f"\ntrades df = {trades.shape}")
trades.info()
print(f"\nfx_daily df = {fx_daily.shape}")
fx_daily.info()

print(f"\ntrades df\n{trades['profit_usd'].describe()}")
print(f"\nclient df by country\n{clients['country'].value_counts()}")
print(f"\ntrade df by pair\n{trades['pair'].value_counts()}")


clients df = (200, 5)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   client_id     200 non-null    int64         
 1   country       200 non-null    object        
 2   account_type  200 non-null    object        
 3   join_date     200 non-null    datetime64[ns]
 4   deposit_usd   200 non-null    int64         
dtypes: datetime64[ns](1), int64(2), object(2)
memory usage: 7.9+ KB

trades df = (3000, 7)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   trade_id    3000 non-null   int64         
 1   date        3000 non-null   datetime64[ns]
 2   client_id   3000 non-null   int64         
 3   pair        3000 non-null   object        
 4   side        3000 non-null   obje

In [3]:
print(f"\n{trades[trades['profit_usd'] > 0]}")
print(f"\n{trades[(trades['pair'] == 'XAUUSD') & (trades['lots'] >= 1.0)]}")
print(f"\n{clients[clients['country'].isin(['Thailand', 'Malaysia'])]}")
print(f"\n{trades.nlargest(10, 'profit_usd')}")
print(f"\n{fx_daily[(fx_daily['pair'] == 'EURUSD') & (fx_daily['date'].dt.year == 2026) & (fx_daily['date'].dt.month == 1)]}")


      trade_id       date  client_id    pair  side  lots  profit_usd
0         2905 2025-07-01         89  XAUUSD   buy  1.00       74.97
1         1713 2025-07-01         27  EURUSD   buy  2.00       69.01
2          730 2025-07-01         58  GBPUSD  sell  0.05        0.17
3         2438 2025-07-01        197  EURUSD  sell  1.00      241.01
4         2623 2025-07-01          9  EURUSD   buy  0.10        1.18
...        ...        ...        ...     ...   ...   ...         ...
2992      2467 2026-06-30        120  XAUUSD   buy  0.50       97.86
2993       739 2026-06-30         40  XAUUSD   buy  0.10       24.30
2995      1903 2026-06-30        102  EURUSD   buy  0.01        1.05
2996      1784 2026-06-30        100  USDJPY  sell  0.05        0.78
2997        31 2026-06-30        144  GBPUSD   buy  0.01        0.95

[1734 rows x 7 columns]

      trade_id       date  client_id    pair  side  lots  profit_usd
0         2905 2025-07-01         89  XAUUSD   buy   1.0       74.97
14     

In [4]:
trades.groupby("pair")["profit_usd"].sum()

pair
EURUSD    5148.46
GBPUSD    4741.21
USDJPY    3867.33
XAUUSD    3504.20
Name: profit_usd, dtype: float64

In [5]:
trades.groupby("pair").agg(
    total_profit = ("profit_usd", "sum"),
    avg_profit = ("profit_usd", "mean"),
    total_trades = ("profit_usd", "count"),
    total_lots = ("lots", "sum")
).sort_values(["total_lots", "total_trades"], ascending=[False, True])

,total_profit,avg_profit,total_trades,total_lots
pair,,,,
XAUUSD,3504.20,4.545006,771,252.96
USDJPY,3867.33,5.371292,720,237.32
EURUSD,5148.46,6.801136,757,232.64
GBPUSD,4741.21,6.304801,752,226.86


In [6]:
trades.groupby(["pair", "side"])["profit_usd"].mean()

pair    side
EURUSD  buy     7.121931
        sell    6.481187
GBPUSD  buy     4.700273
        sell    7.826192
USDJPY  buy     1.394448
        sell    8.831662
XAUUSD  buy     3.477220
        sell    5.757729
Name: profit_usd, dtype: float64

In [7]:
trades.pivot_table(index="pair", columns="side", values="profit_usd", aggfunc="mean")

side,buy,sell
pair,,
EURUSD,7.121931,6.481187
GBPUSD,4.700273,7.826192
USDJPY,1.394448,8.831662
XAUUSD,3.477220,5.757729


In [13]:
trades["pair_avg"] = trades.groupby("pair")["profit_usd"].transform("mean")
print(trades)

      trade_id       date  client_id    pair  side  lots  profit_usd  pair_avg
0         2905 2025-07-01         89  XAUUSD   buy  1.00       74.97  4.545006
1         1713 2025-07-01         27  EURUSD   buy  2.00       69.01  6.801136
2          730 2025-07-01         58  GBPUSD  sell  0.05        0.17  6.304801
3         2438 2025-07-01        197  EURUSD  sell  1.00      241.01  6.801136
4         2623 2025-07-01          9  EURUSD   buy  0.10        1.18  6.801136
...        ...        ...        ...     ...   ...   ...         ...       ...
2995      1903 2026-06-30        102  EURUSD   buy  0.01        1.05  6.801136
2996      1784 2026-06-30        100  USDJPY  sell  0.05        0.78  5.371292
2997        31 2026-06-30        144  GBPUSD   buy  0.01        0.95  6.304801
2998      1282 2026-06-30         56  USDJPY  sell  0.10       -0.62  5.371292
2999       881 2026-06-30        120  XAUUSD  sell  1.00     -130.19  4.545006

[3000 rows x 8 columns]


In [17]:
trades["vs_avg"] = trades["profit_usd"] - trades["pair_avg"]
print(trades)

      trade_id       date  client_id    pair  side  lots  profit_usd  \
0         2905 2025-07-01         89  XAUUSD   buy  1.00       74.97   
1         1713 2025-07-01         27  EURUSD   buy  2.00       69.01   
2          730 2025-07-01         58  GBPUSD  sell  0.05        0.17   
3         2438 2025-07-01        197  EURUSD  sell  1.00      241.01   
4         2623 2025-07-01          9  EURUSD   buy  0.10        1.18   
...        ...        ...        ...     ...   ...   ...         ...   
2995      1903 2026-06-30        102  EURUSD   buy  0.01        1.05   
2996      1784 2026-06-30        100  USDJPY  sell  0.05        0.78   
2997        31 2026-06-30        144  GBPUSD   buy  0.01        0.95   
2998      1282 2026-06-30         56  USDJPY  sell  0.10       -0.62   
2999       881 2026-06-30        120  XAUUSD  sell  1.00     -130.19   

      pair_avg      vs_avg  
0     4.545006   70.424994  
1     6.801136   62.208864  
2     6.304801   -6.134801  
3     6.801136  234

In [20]:
for pair, group in trades.groupby("pair"):
    print(pair, len(group))

EURUSD 757
GBPUSD 752
USDJPY 720
XAUUSD 771


In [25]:
df = trades.merge(clients, on="client_id", how="left")
df.head(10)

,trade_id,date,client_id,pair,side,lots,profit_usd,pair_avg,vs_avg,country,account_type,join_date,deposit_usd
0,2905,2025-07-01,89,XAUUSD,buy,1.00,74.97,4.545006,70.424994,Vietnam,Standard,2025-08-28,500
1,1713,2025-07-01,27,EURUSD,buy,2.00,69.01,6.801136,62.208864,Thailand,cTrader,2026-05-18,2500
2,730,2025-07-01,58,GBPUSD,sell,0.05,0.17,6.304801,-6.134801,Thailand,Standard,2025-09-06,500
3,2438,2025-07-01,197,EURUSD,sell,1.00,241.01,6.801136,234.208864,Singapore,Standard,2025-08-08,5000
4,2623,2025-07-01,9,EURUSD,buy,0.10,1.18,6.801136,-5.621136,Malaysia,Raw Spread,2026-05-21,1000
5,2999,2025-07-01,126,EURUSD,sell,0.10,-16.06,6.801136,-22.861136,Thailand,Standard,2026-01-25,500
6,316,2025-07-01,127,GBPUSD,sell,0.10,9.45,6.304801,3.145199,Philippines,Standard,2025-09-13,500
7,2389,2025-07-01,84,USDJPY,sell,0.10,7.56,5.371292,2.188708,Philippines,Raw Spread,2025-12-25,500
8,1607,2025-07-01,180,EURUSD,sell,0.01,-1.03,6.801136,-7.831136,Thailand,Raw Spread,2026-04-30,500
9,207,2025-07-01,51,USDJPY,buy,0.05,-4.38,5.371292,-9.751292,Philippines,cTrader,2025-10-29,500


In [26]:
df.groupby("account_type")["profit_usd"].mean()

account_type
Raw Spread    6.351820
Standard      5.623302
cTrader       4.825645
Name: profit_usd, dtype: float64

In [29]:
df.groupby(["country", "account_type"])["profit_usd"].sum()

country      account_type
Indonesia    Raw Spread      3045.24
             Standard        1052.99
             cTrader           90.99
Malaysia     Raw Spread      1116.95
             Standard        2798.11
             cTrader          504.71
Philippines  Raw Spread       651.72
             Standard         350.14
             cTrader          400.94
Singapore    Raw Spread       742.91
             Standard        1212.99
             cTrader          -45.96
Thailand     Raw Spread       378.19
             Standard        1782.97
             cTrader         1437.97
Vietnam      Raw Spread       766.16
             Standard        1080.30
             cTrader         -106.12
Name: profit_usd, dtype: float64

In [30]:
df.pivot_table(index="country", columns="account_type", values="profit_usd", aggfunc="sum")

account_type,Raw Spread,Standard,cTrader
country,,,
Indonesia,3045.24,1052.99,90.99
Malaysia,1116.95,2798.11,504.71
Philippines,651.72,350.14,400.94
Singapore,742.91,1212.99,-45.96
Thailand,378.19,1782.97,1437.97
Vietnam,766.16,1080.30,-106.12


In [33]:
pd.crosstab(df["country"], df["pair"])

pair,EURUSD,GBPUSD,USDJPY,XAUUSD
country,,,,
Indonesia,133,123,111,121
Malaysia,185,181,178,193
Philippines,71,75,73,77
Singapore,80,88,76,81
Thailand,188,183,178,181
Vietnam,100,102,104,118


In [34]:
pd.merge(df, fx_daily, on=["date", "pair"])

,trade_id,date,client_id,pair,side,lots,profit_usd,pair_avg,vs_avg,country,account_type,join_date,deposit_usd,open,high,low,close,volume
0,2905,2025-07-01,89,XAUUSD,buy,1.00,74.97,4.545006,70.424994,Vietnam,Standard,2025-08-28,500,2380.00000,2391.76938,2364.57091,2378.17014,19154
1,1713,2025-07-01,27,EURUSD,buy,2.00,69.01,6.801136,62.208864,Thailand,cTrader,2026-05-18,2500,1.08500,1.08784,1.08260,1.08522,58381
2,730,2025-07-01,58,GBPUSD,sell,0.05,0.17,6.304801,-6.134801,Thailand,Standard,2025-09-06,500,1.27000,1.26847,1.26336,1.26591,8468
3,2438,2025-07-01,197,EURUSD,sell,1.00,241.01,6.801136,234.208864,Singapore,Standard,2025-08-08,5000,1.08500,1.08784,1.08260,1.08522,58381
4,2623,2025-07-01,9,EURUSD,buy,0.10,1.18,6.801136,-5.621136,Malaysia,Raw Spread,2026-05-21,1000,1.08500,1.08784,1.08260,1.08522,58381
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2995,1903,2026-06-30,102,EURUSD,buy,0.01,1.05,6.801136,-5.751136,Thailand,Raw Spread,2026-01-13,500,0.96198,0.96446,0.96018,0.96232,43854
2996,1784,2026-06-30,100,USDJPY,sell,0.05,0.78,5.371292,-4.591292,Thailand,cTrader,2026-04-23,1000,168.56463,169.27669,168.08074,168.67871,13723
2997,31,2026-06-30,144,GBPUSD,buy,0.01,0.95,6.304801,-5.354801,Malaysia,cTrader,2026-03-03,500,1.33665,1.33893,1.33691,1.33792,8143
2998,1282,2026-06-30,56,USDJPY,sell,0.10,-0.62,5.371292,-5.991292,Malaysia,Raw Spread,2026-04-10,500,168.56463,169.27669,168.08074,168.67871,13723
